In [ ]:
!nvidia-smi

In [ ]:
%%writefile naive_gemm.cuh
__global__ void naiveGemm(float *A, float *B, float *C, int M, int K, int N) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row < M && col < N) {
        float sum = 0.f;
        for (int k = 0; k < K; k++)
            sum += A[row * K + k] * B[k * N + col];
        C[row * N + col] = sum;
    }
}

In [ ]:
%%writefile naive_gemm.cu
#include <stdio.h>
#include <cuda_runtime.h>
#include "naive_gemm.cuh"

int main() {
    int M = 1024, K = 1024, N = 1024;
    float *A, *B, *C;
    cudaMalloc(&A, M*K*4); cudaMalloc(&B, K*N*4); cudaMalloc(&C, M*N*4);
    dim3 blk(16, 16), grd((N+15)/16, (M+15)/16);
    naiveGemm<<<grd, blk>>>(A, B, C, M, K, N);
    cudaDeviceSynchronize();
    cudaEvent_t s, e; cudaEventCreate(&s); cudaEventCreate(&e);
    int reps = 10; cudaEventRecord(s);
    for (int i = 0; i < reps; i++) naiveGemm<<<grd, blk>>>(A, B, C, M, K, N);
    cudaEventRecord(e); cudaEventSynchronize(e);
    float ms; cudaEventElapsedTime(&ms, s, e);
    printf("Naive GEMM: %.2f TFLOPS  %.2f ms/iter\n",
           2.0*M*K*N*reps/(ms*1e9), ms/reps);
    cudaFree(A); cudaFree(B); cudaFree(C);
}

In [ ]:
!nvcc -arch=sm_75 -O2 -o naive_gemm naive_gemm.cu && ./naive_gemm

In [ ]:
%%writefile tiled_gemm.cuh
#define TILE 16
__global__ void tiledGemm(float *A, float *B, float *C, int M, int K, int N) {
    __shared__ float As[TILE][TILE], Bs[TILE][TILE];
    int row = blockIdx.y*TILE + threadIdx.y;
    int col = blockIdx.x*TILE + threadIdx.x;
    float sum = 0.f;
    for (int t = 0; t < (K+TILE-1)/TILE; t++) {
        As[threadIdx.y][threadIdx.x] = (row < M && t*TILE+threadIdx.x < K)
            ? A[row*K + t*TILE+threadIdx.x] : 0.f;
        Bs[threadIdx.y][threadIdx.x] = (t*TILE+threadIdx.y < K && col < N)
            ? B[(t*TILE+threadIdx.y)*N+col] : 0.f;
        __syncthreads();
        #pragma unroll
        for (int k = 0; k < TILE; k++) sum += As[threadIdx.y][k]*Bs[k][threadIdx.x];
        __syncthreads();
    }
    if (row < M && col < N) C[row*N+col] = sum;
}

In [ ]:
%%writefile compare_gemm.cu
#include <stdio.h>
#include <cuda_runtime.h>
#include "naive_gemm.cuh"
#include "tiled_gemm.cuh"

static void bench(const char *name, float *A, float *B, float *C,
                  int M, int K, int N, int tiled) {
    dim3 blk(16,16), grd((N+15)/16,(M+15)/16);
    if (tiled) tiledGemm<<<grd,blk>>>(A,B,C,M,K,N);
    else        naiveGemm<<<grd,blk>>>(A,B,C,M,K,N);
    cudaDeviceSynchronize();
    cudaEvent_t s, e; cudaEventCreate(&s); cudaEventCreate(&e);
    int reps = 20; cudaEventRecord(s);
    for (int i = 0; i < reps; i++) {
        if (tiled) tiledGemm<<<grd,blk>>>(A,B,C,M,K,N);
        else        naiveGemm<<<grd,blk>>>(A,B,C,M,K,N);
    }
    cudaEventRecord(e); cudaEventSynchronize(e);
    float ms; cudaEventElapsedTime(&ms,s,e);
    printf("%-12s %.2f TFLOPS  %.2f ms\n", name,
           2.0*M*K*N*reps/(ms*1e9), ms/reps);
    cudaEventDestroy(s); cudaEventDestroy(e);
}

int main() {
    int M = 2048, K = 2048, N = 2048;
    float *A, *B, *C;
    cudaMalloc(&A,M*K*4); cudaMalloc(&B,K*N*4); cudaMalloc(&C,M*N*4);
    bench("Naive", A,B,C,M,K,N,0);
    bench("Tiled", A,B,C,M,K,N,1);
    cudaFree(A); cudaFree(B); cudaFree(C);
}

In [ ]:
!nvcc -arch=sm_75 -O2 -o compare compare_gemm.cu && ./compare

In [ ]:
import torch
M, K, N = 2048, 2048, 2048
A = torch.randn(M, K, device='cuda')
B = torch.randn(K, N, device='cuda')
torch.mm(A, B)
s = torch.cuda.Event(enable_timing=True)
e = torch.cuda.Event(enable_timing=True)
reps = 20
s.record()
for _ in range(reps): torch.mm(A, B)
e.record(); torch.cuda.synchronize()
ms = s.elapsed_time(e) / reps
print(f"torch.mm:    {2*M*K*N/ms/1e9:.2f} TFLOPS  {ms:.2f} ms")